# Stage A — Building the corpus
## From database exports to `corpus.csv`

**Companion to** `02_cognitive_analysis.ipynb`
Lartey & Law (2026), *Technology in Society* 86, 103321

---

This notebook turns raw RIS exports from Scopus and ScienceDirect into the single
`data/corpus.csv` that the analysis notebook reads. It covers Section 3.1 of the paper:
search, deduplication, screening, and indicator tagging.

### Bring your own exports

The corpus behind the published paper is not distributed here. This notebook is
written so that you can run the same pipeline over **your own literature search**:
drop your `.ris` exports into `data/raw_ris/`, run the stages in order, and you get a
`corpus.csv` with the structure the analysis notebook expects.

All paths are relative to the repository, so nothing needs editing before you start.

### The output contract

This notebook writes `data/corpus.csv` with exactly the columns notebook 02 requires:
`Title`, `Abstract`, `Keyword`, `Year`, `SearchIndicator`, plus `DOI` and `Source`.

---
## A0 — Setup

Put your `.ris` exports in `data/raw_ris/`. One file per search string keeps the
indicator tagging in Stage A4 simple, but a mixed folder also works.

In [ ]:
from __future__ import annotations

import re
import json
from collections import Counter
from pathlib import Path

import pandas as pd

def _repo_relative(p):
    """Resolve whether the kernel started in notebooks/ or the repository root."""
    p = Path(p)
    if p.exists():
        return p
    alt = Path(str(p).replace("../", "", 1))
    return alt if (alt.exists() or Path("notebooks").is_dir()) else p


DATA_DIR = _repo_relative("../data")
RIS_DIR = DATA_DIR / "raw_ris"
INTERIM = DATA_DIR / "interim"
for d in (DATA_DIR, RIS_DIR, INTERIM):
    d.mkdir(parents=True, exist_ok=True)

# The 12 indicators. Must match notebook 02 exactly.
INDICATORS = ["Action", "Agency", "Culture", "Data", "Governance", "Materiality",
              "Personality", "Security", "Space", "Sustainability", "Technology", "Time"]

# Search strings, Table 1 in the paper. Also used for fallback tagging in A4.
SEARCH_STRINGS = {
    "Technology":     ["AIoT", "autonomous infrastructure", "machine learning",
                       "urban AI technology"],
    "Action":         ["AI decision-making", "predictive analytics", "algorithmic action",
                       "urban interventions"],
    "Space":          ["spatial governance", "urban morphologies", "smart urban spaces",
                       "digital twin cities"],
    "Data":           ["datafication", "algorithmic governance", "urban data ethics",
                       "surveillance"],
    "Sustainability": ["AI for SDGs", "climate-smart cities", "resilient infrastructure",
                       "urban decarbonization"],
    "Security":       ["AI surveillance", "predictive policing", "cybersecurity in cities",
                       "AI risk mitigation"],
    "Agency":         ["human-AI interaction", "distributed agency", "AI as actor",
                       "urban sociotechnical systems"],
    "Culture":        ["AI imaginaries", "AI narratives", "cultural urbanism",
                       "future cities discourse"],
    "Personality":    ["AI personality", "chatbots", "robots in governance",
                       "affective computing", "human-centered AI"],
    "Governance":     ["AI regulation", "smart city governance", "AI ethics",
                       "governing AI in urban planning"],
    "Time":           ["temporal AI", "future-oriented governance", "urban foresight",
                       "AI and planning horizons"],
    "Materiality":    ["AI hardware", "urban infrastructure", "data centers",
                       "material politics of AI"],
}

ris_files = sorted(RIS_DIR.glob("*.ris"))
print(f"RIS folder: {RIS_DIR.resolve()}")
print(f"Files found: {len(ris_files)}")
for f in ris_files:
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")
if not ris_files:
    print("\nNo .ris files yet. Drop your exports into data/raw_ris/ and re-run.")

---
## A1 — Parse RIS

RIS is line-oriented: each line is `TAG  - value`, continuation lines are indented or
untagged, and `ER  -` closes a record.

The previous parser did `ris_text.replace("\n", " ")` and then applied a regex looking
for `[A-Z][A-Z0-9]  - `. Two failure modes follow from that. Multi-line abstracts get
folded into one line, which is survivable; but any abstract containing a pattern like
`AI  - ` mid-sentence is read as a new field, silently truncating the record. Records
also lose their boundaries if `ER` is missing.

The parser below reads line by line, keeps continuations attached to their field, and
counts anything it cannot interpret so nothing is dropped quietly.

In [ ]:
RIS_FIELDS = {
    "TY": "Type", "T1": "Title", "TI": "Title", "AU": "Authors", "JO": "Journal",
    "JF": "Journal", "T2": "Journal", "PY": "Year", "DA": "Date", "AB": "Abstract",
    "N2": "Abstract", "KW": "Keyword", "DO": "DOI", "UR": "URL", "SN": "ISSN",
    "VL": "Volume", "IS": "Issue", "SP": "StartPage", "EP": "EndPage", "PB": "Publisher",
}
MULTI = {"AU", "KW"}
LINE = re.compile(r"^([A-Z][A-Z0-9])\s{2}-\s?(.*)$")


def parse_ris(path):
    """Line-oriented RIS parser. Returns (records, stats)."""
    records, record, last_tag = [], {}, None
    stats = Counter()

    with open(path, "r", encoding="utf-8-sig", errors="replace") as fh:
        for raw in fh:
            line = raw.rstrip("\r\n")
            if not line.strip():
                continue
            m = LINE.match(line)
            if m:
                tag, value = m.group(1), m.group(2).strip()
                if tag == "ER":
                    if record:
                        records.append(record)
                        stats["records"] += 1
                    record, last_tag = {}, None
                    continue
                if tag == "TY" and record:
                    # A new record started without a closing ER - keep the old one.
                    records.append(record)
                    stats["records"] += 1
                    stats["missing_ER"] += 1
                    record = {}
                name = RIS_FIELDS.get(tag)
                if name is None:
                    stats[f"unmapped:{tag}"] += 1
                    last_tag = None
                    continue
                if tag in MULTI:
                    record[name] = f"{record[name]}; {value}" if name in record else value
                else:
                    record.setdefault(name, value)
                last_tag = name
            elif last_tag:
                # Continuation of the previous field, e.g. a wrapped abstract.
                record[last_tag] = f"{record[last_tag]} {line.strip()}".strip()
                stats["continuation_lines"] += 1
            else:
                stats["orphan_lines"] += 1

    if record:
        records.append(record)
        stats["records"] += 1
        stats["missing_ER"] += 1
    return records, stats


frames, all_stats = [], {}
for f in ris_files:
    recs, st = parse_ris(f)
    if not recs:
        print(f"  {f.name}: no records parsed")
        continue
    df = pd.DataFrame(recs)
    df["SourceFile"] = f.name
    frames.append(df)
    all_stats[f.name] = dict(st)
    print(f"  {f.name}: {st['records']:>4} records, "
          f"{st['continuation_lines']} continuation lines, "
          f"{st['orphan_lines']} orphan lines")

if frames:
    raw = pd.concat(frames, ignore_index=True)
    raw.to_csv(INTERIM / "01_parsed.csv", index=False)
    print(f"\nParsed {len(raw):,} records -> {INTERIM / '01_parsed.csv'}")
    display(raw.head(3))
else:
    raw = pd.DataFrame()
    print("\nNothing parsed. Add .ris files to data/raw_ris/ and re-run A0 and A1.")

---
## A2 — Normalise fields

Map RIS tags onto the schema notebook 02 expects. Year is taken from `PY` and falls
back to the first four-digit number in `DA`; DOIs are stripped to the bare identifier
so that deduplication in A3 does not treat `10.1016/x` and
`https://doi.org/10.1016/x` as two different papers.

In [ ]:
def normalise(df):
    if df.empty:
        return df
    out = pd.DataFrame(index=df.index)
    out["Title"] = df.get("Title", "").fillna("").astype(str).str.strip()
    out["Abstract"] = df.get("Abstract", "").fillna("").astype(str).str.strip()
    out["Keyword"] = df.get("Keyword", "").fillna("").astype(str).str.strip()
    out["Authors"] = df.get("Authors", "").fillna("").astype(str).str.strip()
    out["Journal"] = df.get("Journal", "").fillna("").astype(str).str.strip()

    year = pd.to_numeric(df.get("Year"), errors="coerce")
    if "Date" in df.columns:
        fallback = pd.to_numeric(
            df["Date"].astype(str).str.extract(r"(\d{4})")[0], errors="coerce")
        year = year.fillna(fallback)
    out["Year"] = year

    doi = df.get("DOI", "").fillna("").astype(str).str.strip().str.lower()
    out["DOI"] = (doi.str.replace(r"^https?://(dx\.)?doi\.org/", "", regex=True)
                     .str.replace(r"^doi:\s*", "", regex=True).str.strip())
    out["URL"] = df.get("URL", "").fillna("").astype(str)
    out["SourceFile"] = df.get("SourceFile", "").fillna("").astype(str)
    return out


records = normalise(raw)
if not records.empty:
    print(f"Records: {len(records):,}")
    print(f"  with a DOI:      {(records['DOI'] != '').sum():,}")
    print(f"  with an abstract:{(records['Abstract'] != '').sum():,}")
    print(f"  with a year:     {records['Year'].notna().sum():,}")
    if records["Year"].notna().any():
        print(f"  year range:      {int(records['Year'].min())}-{int(records['Year'].max())}")
    records.to_csv(INTERIM / "02_normalised.csv", index=False)
    display(records.head(3))

---
## A3 — Deduplicate

Two passes, in order. DOI is exact and reliable, so it goes first. Records without a
DOI fall through to a normalised-title match: lowercased, punctuation stripped,
whitespace collapsed. Every removal is counted, because the paper reports **n = 412**
duplicates and that number should be checkable.

In [ ]:
def norm_title(s):
    return (s.astype(str).str.lower()
             .str.replace(r"[^a-z0-9 ]", " ", regex=True)
             .str.replace(r"\s+", " ", regex=True).str.strip())


ledger = []

if not records.empty:
    n0 = len(records)
    d = records.copy()

    has_doi = d["DOI"] != ""
    dup_doi = d[has_doi].duplicated(subset=["DOI"], keep="first")
    drop_doi = d[has_doi].index[dup_doi]
    d = d.drop(index=drop_doi)
    ledger.append({"step": "Deduplicate by DOI", "removed": len(drop_doi), "remaining": len(d)})

    d["_t"] = norm_title(d["Title"])
    dup_title = d["_t"].ne("") & d.duplicated(subset=["_t"], keep="first")
    n_title = int(dup_title.sum())
    d = d[~dup_title].drop(columns="_t")
    ledger.append({"step": "Deduplicate by title", "removed": n_title, "remaining": len(d)})

    deduped = d.reset_index(drop=True)
    print(f"Started with {n0:,}")
    print(f"  removed {len(drop_doi):,} DOI duplicates")
    print(f"  removed {n_title:,} title duplicates")
    print(f"Remaining: {len(deduped):,}   (paper reports 412 duplicates removed)")
    deduped.to_csv(INTERIM / "03_deduplicated.csv", index=False)
    display(pd.DataFrame(ledger))

---
## A4 — Tag the search indicator

Every record needs a `SearchIndicator`: which of the twelve conceptual categories its
search string belongs to. This is the column the analysis notebook groups everything
by, so it is worth getting right rather than accepting whatever the fallback produces.

Two ways to get it, tried in order:

1. **From the filename.** If exports are named per search string — `Technology_1.ris`,
   `governance_b.ris` — the indicator is read straight off the filename. This is exact
   and is the recommended way to organise the exports.
2. **From the text.** Otherwise each record is matched against the Table 1 search
   strings and assigned the indicator with the most term hits, with ties broken by
   the order in `INDICATORS`.

The distribution is printed at the end. Compare it against Figure 1 of the paper: if
one indicator is wildly over- or under-represented, the search strings behind it
probably need attention before you go any further.

In [ ]:
def indicator_from_filename(name):
    stem = Path(name).stem.lower()
    for ind in INDICATORS:
        if re.search(rf"(^|[^a-z]){ind.lower()}([^a-z]|$)", stem):
            return ind
    return None


def indicator_from_text(text, strings=SEARCH_STRINGS):
    t = text.lower()
    scores = {ind: sum(t.count(term.lower()) for term in terms)
              for ind, terms in strings.items()}
    best = max(scores.values())
    if best == 0:
        return None
    for ind in INDICATORS:                      # deterministic tie-break
        if scores.get(ind, 0) == best:
            return ind
    return None


if not records.empty:
    tagged = deduped.copy()
    tagged["SearchIndicator"] = tagged["SourceFile"].map(indicator_from_filename)
    n_file = int(tagged["SearchIndicator"].notna().sum())

    need = tagged["SearchIndicator"].isna()
    if need.any():
        blob = (tagged.loc[need, "Title"] + " " + tagged.loc[need, "Abstract"] + " "
                + tagged.loc[need, "Keyword"])
        tagged.loc[need, "SearchIndicator"] = blob.map(indicator_from_text)

    n_text = int(tagged["SearchIndicator"].notna().sum()) - n_file
    n_none = int(tagged["SearchIndicator"].isna().sum())

    print(f"Tagged from filename: {n_file:,}")
    print(f"Tagged from text:     {n_text:,}")
    print(f"Untagged:             {n_none:,}")
    if n_file == 0:
        print("\n  No filename matched an indicator. Naming your exports after the")
        print("  search string (e.g. Governance_1.ris) makes this step exact.")
    display(tagged["SearchIndicator"].value_counts(dropna=False)
            .rename("records").to_frame())

---
## A5 — Screen and export

Inclusion criteria from Section 3.1: engages with AI **and** with urban systems or
governance, has a title, has a usable year.

Each rule records how many records it removed, so the funnel can be read off directly
rather than reconstructed. The paper reports 7,354 initial records, 412 duplicates,
1,128 excluded at title/abstract, 180 at full text, leaving 5,634. Full-text screening
is a human step and is not automated here — the ledger leaves a row for you to fill in
once it is done.

In [ ]:
AI_TERMS = ["artificial intelligence", "machine learning", "deep learning",
            "algorithm", "neural network", "autonomous", " ai ", "predictive",
            "automation", "chatbot", "computer vision", "llm", "generative"]
URBAN_TERMS = ["urban", "city", "cities", "municipal", "smart city", "governance",
               "planning", "metropolitan", "civic", "public sector", "infrastructure"]


def contains_any(series, terms):
    pattern = "|".join(re.escape(t) for t in terms)
    return series.str.lower().str.contains(pattern, regex=True, na=False)


if not records.empty:
    s = tagged.copy()
    blob = (s["Title"] + " " + s["Abstract"] + " " + s["Keyword"])

    n = len(s)
    s = s[s["Title"].str.len() > 10]
    ledger.append({"step": "Title present", "removed": n - len(s), "remaining": len(s)})

    n = len(s)
    s = s[s["Year"].notna() & s["Year"].between(1900, 2100)]
    ledger.append({"step": "Usable year", "removed": n - len(s), "remaining": len(s)})

    blob = (s["Title"] + " " + s["Abstract"] + " " + s["Keyword"])
    n = len(s)
    s = s[contains_any(blob, AI_TERMS)]
    ledger.append({"step": "Mentions AI", "removed": n - len(s), "remaining": len(s)})

    blob = (s["Title"] + " " + s["Abstract"] + " " + s["Keyword"])
    n = len(s)
    s = s[contains_any(blob, URBAN_TERMS)]
    ledger.append({"step": "Mentions urban/governance", "removed": n - len(s),
                   "remaining": len(s)})

    n = len(s)
    s = s[s["SearchIndicator"].notna()]
    ledger.append({"step": "Indicator assigned", "removed": n - len(s), "remaining": len(s)})

    ledger.append({"step": "Full-text screening (manual)", "removed": "TODO",
                   "remaining": "TODO"})

    screening = pd.DataFrame(ledger)
    screening.to_csv(INTERIM / "04_screening_ledger.csv", index=False)
    print("Screening funnel")
    display(screening)

    corpus = s[["Title", "Abstract", "Keyword", "Year", "SearchIndicator",
                "DOI", "Authors", "Journal", "SourceFile"]].copy()
    corpus["Year"] = corpus["Year"].astype(int)
    corpus = corpus.reset_index(drop=True)

    out = DATA_DIR / "corpus.csv"
    corpus.to_csv(out, index=False)
    print(f"\nWrote {len(corpus):,} records to {out.resolve()}")
    print("\nOpen 02_cognitive_analysis.ipynb and set CONFIG['data_mode'] = 'real'.")
    display(corpus.head(3))

---
## A6 — Check the handover

A last check that `corpus.csv` satisfies notebook 02's contract, so that a schema
problem surfaces here rather than four stages into the analysis.

In [ ]:
REQUIRED = ["Title", "Abstract", "Keyword", "Year", "SearchIndicator"]
path = DATA_DIR / "corpus.csv"

if not path.exists():
    print(f"{path} does not exist yet. Run A0 to A5 first.")
else:
    c = pd.read_csv(path)
    ok = True

    missing = [col for col in REQUIRED if col not in c.columns]
    print(f"[{'PASS' if not missing else 'FAIL'}] required columns present")
    if missing:
        print(f"        missing: {missing}")
        ok = False

    if not missing:
        bad = set(c["SearchIndicator"].dropna().unique()) - set(INDICATORS)
        print(f"[{'PASS' if not bad else 'FAIL'}] SearchIndicator values recognised")
        if bad:
            print(f"        unexpected: {sorted(bad)}")
            ok = False

        yr = pd.to_numeric(c["Year"], errors="coerce")
        bad_yr = int((yr.isna() | ~yr.between(1900, 2100)).sum())
        print(f"[{'PASS' if not bad_yr else 'WARN'}] years parse and fall in range "
              f"({bad_yr} problem rows)")

        blank = c["Abstract"].isna().mean()
        print(f"[{'PASS' if blank < 0.5 else 'WARN'}] abstracts populated "
              f"({blank:.0%} blank)")

        absent = set(INDICATORS) - set(c["SearchIndicator"].dropna().unique())
        print(f"[{'PASS' if not absent else 'WARN'}] all 12 indicators represented")
        if absent:
            print(f"        no records for: {sorted(absent)}")

    print("\n" + ("Ready for 02_cognitive_analysis.ipynb."
                   if ok else "Fix the FAIL lines above before continuing."))
    print(f"Records: {len(c):,}")

---

Next: **`02_cognitive_analysis.ipynb`**.

Citation: Lartey, D. & Law, K. M. Y. (2026). Governing with artificial intelligence:
Mapping the knowledge systems shaping urban intelligence. *Technology in Society*, 86,
103321. [10.1016/j.techsoc.2026.103321](https://doi.org/10.1016/j.techsoc.2026.103321)